[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/chain.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58238466-lesson-4-chain)

# Chain

## Review

We built a simple graph with nodes, normal edges, and conditional edges.

## Goals

Now, let's build up to a simple chain that combines 4 [concepts](https://python.langchain.com/v0.2/docs/concepts/):

* Using [chat messages](https://python.langchain.com/v0.2/docs/concepts/#messages) as our graph state
* Using [chat models](https://python.langchain.com/v0.2/docs/concepts/#chat-models) in graph nodes
* [Binding tools](https://python.langchain.com/v0.2/docs/concepts/#tools) to our chat model
* [Executing tool calls](https://python.langchain.com/v0.2/docs/concepts/#functiontool-calling) in graph nodes 

![Screenshot 2024-08-21 at 9.24.03 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbab08dd607b08df5e1101_chain1.png)

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langgraph

## Messages

Chat models can use [`messages`](https://python.langchain.com/v0.2/docs/concepts/#messages), which capture different roles within a conversation. 

LangChain supports various message types, including `HumanMessage`, `AIMessage`, `SystemMessage`, and `ToolMessage`. 

These represent a message from the user, from chat model, for the chat model to instruct behavior, and from a tool call. 

Let's create a list of messages. 

Each message can be supplied with a few things:

* `content` - content of the message
* `name` - optionally, a message author 
* `response_metadata` - optionally, a dict of metadata (e.g., often populated by model provider for `AIMessages`)

In [5]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage

messages = [AIMessage(content=f"So you said you were researching ocean mammals?", name="Model")]
messages.append(HumanMessage(content=f"Yes, that's right.",name="Lance"))
messages.append(AIMessage(content=f"Great, what would you like to learn about.", name="Model"))
messages.append(HumanMessage(content=f"I want to learn about the best place to see Orcas in the US.", name="Lance"))

for m in messages:
    m.pretty_print()

================================== Ai Message ==================================
Name: Model

So you said you were researching ocean mammals?
================================ Human Message =================================
Name: Lance

Yes, that's right.
================================== Ai Message ==================================
Name: Model

Great, what would you like to learn about.
================================ Human Message =================================
Name: Lance

I want to learn about the best place to see Orcas in the US.


## Chat Models

[Chat models](https://python.langchain.com/v0.2/docs/concepts/#chat-models) can use a sequence of message as input and support message types, as discussed above.

There are [many](https://python.langchain.com/v0.2/docs/concepts/#chat-models) to choose from! Let's work with OpenAI. 

Let's check that your `GROQ_API_KEY` is set and, if not, you will be asked to enter it.

In [6]:
import sys
import os
from pathlib import Path

# Get current working directory and go up one level
current_dir = Path.cwd()
parent_dir = current_dir.parent
sys.path.append(str(parent_dir))

# Now import should work
from utils import setup_environment
# Or specify your required variables
setup_environment(["LANGSMITH_API_KEY", "GROQ_API_KEY", "TAVILY_API_KEY"], env_file="../.env")



🚀 Setting up environment variables...
Loading environment variables from ../.env
✅ Environment variables loaded from .env file

📋 Checking required environment variables:
✅ LANGSMITH_API_KEY already configured
✅ GROQ_API_KEY already configured
✅ TAVILY_API_KEY already configured

🎉 Environment setup complete!


We can load a chat model and invoke it with out list of messages.

We can see that the result is an `AIMessage` with specific `response_metadata`.

In [16]:
# Free Groq models instead of OpenAI
from langchain_groq import ChatGroq

# Groq's powerful models (free tier)
# mixtral_chat = ChatGroq(model="mixtral-8x7b-32768", temperature=0)
# llama_chat = ChatGroq(model="llama2-70b-4096", temperature=0)
# Alternative models available on Groq
# gemma_chat = ChatGroq(model="gemma-7b-it", temperature=0)
# llama3_chat = ChatGroq(model="llama3-8b-8192", temperature=0)  # If available

llm1 = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm2 = ChatGroq(model="qwen/qwen3-32b", temperature=0)

result = llm1.invoke(messages)
type(result)

langchain_core.messages.ai.AIMessage

In [17]:
result

AIMessage(content='Orcas, also known as killer whales, are an exciting sight to see in their natural habitat. In the US, some of the best places to see orcas include:\n\n1. **San Juan Islands, Washington**: This is one of the most popular and reliable places to see orcas in the US. The San Juan Islands are home to a large population of southern resident orcas, and you can take a guided tour to increase your chances of spotting them.\n2. **Puget Sound, Washington**: Similar to the San Juan Islands, Puget Sound is another great spot to see orcas in Washington state. You can take a ferry or a guided tour to explore the sound and potentially spot orcas.\n3. **Monterey Bay, California**: Monterey Bay is known for its diverse marine life, including orcas. You can take a guided tour or visit the Monterey Bay Aquarium to learn more about these amazing creatures.\n4. **Alaska**: Alaska is home to a large population of orcas, and you can see them in various locations, including Juneau, Seward, a

In [18]:
result.response_metadata

{'token_usage': {'completion_tokens': 443,
  'prompt_tokens': 91,
  'total_tokens': 534,
  'completion_time': 0.965025341,
  'prompt_time': 0.004812622,
  'queue_time': 0.103021597,
  'total_time': 0.969837963},
 'model_name': 'llama-3.3-70b-versatile',
 'system_fingerprint': 'fp_155ab82e98',
 'service_tier': 'on_demand',
 'finish_reason': 'stop',
 'logprobs': None}

## Tools

Tools are useful whenever you want a model to interact with external systems.

External systems (e.g., APIs) often require a particular input schema or payload, rather than natural language. 

When we bind an API, for example, as a tool we given the model awareness of the required input schema.

The model will choose to call a tool based upon the natural language input from the user. 

And, it will return an output that adheres to the tool's schema. 

[Many LLM providers support tool calling](https://python.langchain.com/v0.1/docs/integrations/chat/) and [tool calling interface](https://blog.langchain.dev/improving-core-tool-interfaces-and-docs-in-langchain/) in LangChain is simple. 
 
You can simply pass any Python `function` into `ChatModel.bind_tools(function)`.

![Screenshot 2024-08-19 at 7.46.28 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbab08dc1c17a7a57f9960_chain2.png)

Let's showcase a simple example of tool calling!
 
The `multiply` function is our tool.

In [56]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq

# Correct tool definitions
@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b.
    Args:
        a: first int
        b: second int
    """
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Add a and b.  # ← Fixed docstring
    Args:
        a: first int
        b: second int
    """
    return a + b

# Create LLM and bind tools
llm = ChatGroq(model="llama-3.3-70b-versatile")
llm_with_tools = llm.bind_tools([add, multiply])

# Test the tools
response_tool_call_1 = llm_with_tools.invoke([HumanMessage(content="What is 2 multiplied by 3", name="Lance")])
print("Multiply tool calls:", response_tool_call_1.tool_calls)

response_tool_call_2 = llm_with_tools.invoke([HumanMessage(content="What is 2 plus 3", name="Lance")])
print("Add tool calls:", response_tool_call_2.tool_calls)

# Also check the content
print("Multiply response:", response_tool_call_1.content)
print("Add response:", response_tool_call_2.content)

Multiply tool calls: [{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'vdcqfrj85', 'type': 'tool_call'}]
Add tool calls: [{'name': 'add', 'args': {'a': 2, 'b': 3}, 'id': 'jhs197cec', 'type': 'tool_call'}]
Multiply response: 
Add response: 


If we pass an input - e.g., `"What is 2 multiplied by 3"` - we see a tool call returned. 

The tool call has specific arguments that match the input schema of our function along with the name of the function to call.

```
{'arguments': '{"a":2,"b":3}', 'name': 'multiply'}
```

In [57]:
response_tool_call_1

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'vdcqfrj85', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 352, 'total_tokens': 371, 'completion_time': 0.059009142, 'prompt_time': 0.022278355, 'queue_time': 0.095337244, 'total_time': 0.081287497}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_9e1e8f8435', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--0cf76202-2374-4f24-a65f-801c2ce1577c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'vdcqfrj85', 'type': 'tool_call'}], usage_metadata={'input_tokens': 352, 'output_tokens': 19, 'total_tokens': 371})

In [58]:
response_tool_call_2

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jhs197cec', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 351, 'total_tokens': 369, 'completion_time': 0.035517095, 'prompt_time': 0.029579029, 'queue_time': 0.102261953, 'total_time': 0.065096124}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_9e1e8f8435', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--ed126abc-81c3-44c5-bcf0-42253557d673-0', tool_calls=[{'name': 'add', 'args': {'a': 2, 'b': 3}, 'id': 'jhs197cec', 'type': 'tool_call'}], usage_metadata={'input_tokens': 351, 'output_tokens': 18, 'total_tokens': 369})

In [59]:
response_tool_call_1.tool_calls[0]["args"]

{'a': 2, 'b': 3}

In [60]:
response_tool_call_1.additional_kwargs

{'tool_calls': [{'id': 'vdcqfrj85',
   'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'},
   'type': 'function'}]}

## Using messages as state

With these foundations in place, we can now use [`messages`](https://python.langchain.com/v0.2/docs/concepts/#messages) in our graph state.

Let's define our state, `MessagesState`, as a `TypedDict` with a single key: `messages`.

`messages` is simply a list of messages, as we defined above (e.g., `HumanMessage`, etc).

In [63]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage

class MessagesState(TypedDict):
    messages: list[AnyMessage]

## Reducers

Now, we have a minor problem! 

As we discussed, each node will return a new value for our state key `messages`.

But, this new value [will override](https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers) the prior `messages` value.
 
As our graph runs, we want to **append** messages to our `messages` state key.
 
We can use [reducer functions](https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers) to address this.

Reducers allow us to specify how state updates are performed.

If no reducer function is specified, then it is assumed that updates to the key should *override it* as we saw before.
 
But, to append messages, we can use the pre-built `add_messages` reducer.

This ensures that any messages are appended to the existing list of messages.

We simply need to annotate our `messages` key with the `add_messages` reducer function as metadata.

In [69]:
from typing import Annotated
from langgraph.graph.message import add_messages

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

Since having a list of messages in graph state is so common, LangGraph has a pre-built [`MessagesState`](https://langchain-ai.github.io/langgraph/concepts/low_level/#messagesstate)! 

`MessagesState` is defined: 

* With a pre-build single `messages` key
* This is a list of `AnyMessage` objects 
* It uses the `add_messages` reducer

We'll usually use `MessagesState` because it is less verbose than defining a custom `TypedDict`, as shown above.

In [70]:
from langgraph.graph import MessagesState

# class MessagesState(MessagesState):
#     # Add any keys needed beyond messages, which is pre-built 
#     pass

To go a bit deeper, we can see how the `add_messages` reducer works in isolation.

In [72]:
# Initial state
initial_messages = [
    AIMessage(content="Hello! How can I assist you?", name="Model"),
    HumanMessage(content="I'm looking for information on marine biology.", name="Lance")
]

# New message to add
new_message = AIMessage(
    content="Sure, I can help with that. What specifically are you interested in?", 
    name="Model",
)

# Test
add_messages(initial_messages , new_message)

[AIMessage(content='Hello! How can I assist you?', additional_kwargs={}, response_metadata={}, name='Model', id='5412a7e0-4c64-4146-ba45-1f5a0220d2f1'),
 HumanMessage(content="I'm looking for information on marine biology.", additional_kwargs={}, response_metadata={}, name='Lance', id='60c9fd5c-e5a4-4bd3-a53b-3619891f2310'),
 AIMessage(content='Sure, I can help with that. What specifically are you interested in?', additional_kwargs={}, response_metadata={}, name='Model', id='004376d8-c1aa-45b0-8033-8be1479dc043')]

## Our graph

Now, lets use `MessagesState` with a graph.

In [96]:
# # Install for local rendering
# !pip install playwright
# !playwright install

In [97]:
from langgraph.graph import StateGraph, START, END
    
# Node
def tool_calling_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# Build graph
builder = StateGraph(MessagesState)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_edge(START, "tool_calling_llm")
builder.add_edge("tool_calling_llm", END)
graph = builder.compile()

In [98]:
from langchain_core.runnables.graph_mermaid import MermaidDrawMethod
from IPython.display import Image, display

# # View
# display(Image(graph.get_graph().draw_mermaid_png()))

# # ASCII visualization (always works)
# print("Graph Structure:")
# print(graph.get_graph().draw_ascii())

try:
    # Try local rendering
    display(Image(graph.get_graph().draw_mermaid_png(
        draw_method=MermaidDrawMethod.PYPPETEER
    )))
except Exception as e:
    print(f"Pyppeteer failed: {e}")
    # Fallback to ASCII
    print(graph.get_graph().draw_ascii())

Pyppeteer failed: asyncio.run() cannot be called from a running event loop
    +-----------+    
    | __start__ |    
    +-----------+    
          *          
          *          
          *          
+------------------+ 
| tool_calling_llm | 
+------------------+ 
          *          
          *          
          *          
    +---------+      
    | __end__ |      
    +---------+      


/var/folders/tr/_8gn26jn35x_vmmf6p7s4tlr0000gn/T/ipykernel_77547/253371735.py:19: RuntimeWarning: coroutine '_render_mermaid_using_pyppeteer' was never awaited
  print(graph.get_graph().draw_ascii())


If we pass in `Hello!`, the LLM responds without any tool calls.

In [94]:
messages = graph.invoke({"messages": HumanMessage(content="Hello!")})
for m in messages['messages']:
    m.pretty_print()

================================ Human Message =================================

Hello!
================================== Ai Message ==================================

I'm here to help with any questions or tasks you may have. What's on your mind today?


The LLM chooses to use a tool when it determines that the input or task requires the functionality provided by that tool.

In [95]:
messages = graph.invoke({"messages": HumanMessage(content="Multiply 2 and 3")})
for m in messages['messages']:
    m.pretty_print()

================================ Human Message =================================

Multiply 2 and 3
================================== Ai Message ==================================
Tool Calls:
  multiply (z98sscfsw)
 Call ID: z98sscfsw
  Args:
    a: 2
    b: 3
